# Wholesale Customer Segmentation Using K-Means

## Project Overview
This notebook presents an end-to-end unsupervised learning workflow for segmenting clients of a wholesale distributor based on their annual spending across six product categories:
- `Fresh`
- `Milk`
- `Grocery`
- `Frozen`
- `Detergents_Paper`
- `Delicassen`

Additionally, customer channel (`Channel`: Horeca vs Retail) and geographic region (`Region`: Lisbon, Oporto, Other) are used for post-clustering profiling and strategic business evaluation.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

FEATURE_COLS = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
CATEGORICAL_COLS = ['Channel', 'Region']
CHANNEL_MAP = {1: 'Horeca', 2: 'Retail'}
REGION_MAP = {1: 'Lisbon', 2: 'Oporto', 3: 'Other Region'}


## 1. Data Loading and Data Audit
We load `Wholesale_Customers.csv` and inspect missing values, duplicates, and metadata alignment.


In [ ]:
# Load dataset
data_path = 'Wholesale_Customers.csv' if os.path.exists('Wholesale_Customers.csv') else 'dataset/Wholesale_Customers.csv'
df = pd.read_csv(data_path)
print("Dataset shape:", df.shape)
print("Missing values per column:
", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
df.head()


## 2. Exploratory Data Analysis (EDA)
Examining spending distributions, skewness, kurtosis, and correlation across product categories.


In [ ]:
# Summary statistics
print("Descriptive Statistics (Raw Scale):")
display_df = df[FEATURE_COLS].describe().T[['mean', 'std', 'min', '50%', 'max']]
display_df.rename(columns={'50%': 'median'}, inplace=True)
print(display_df)

print("
Skewness Raw:")
print(df[FEATURE_COLS].skew())
print("
Skewness Log1p Transformed:")
print(np.log1p(df[FEATURE_COLS]).skew())


In [ ]:
# Pearson correlation between spending features
plt.figure(figsize=(8, 6))
sns.heatmap(df[FEATURE_COLS].corr(), annot=True, fmt='.2f', cmap='Blues', square=True)
plt.title('Correlation Matrix of Spending Categories (Raw Scale)')
plt.show()


## 3. Preprocessing & Scaling Strategy
Given the severe right-skewness and heavy tails, we apply:
1. `log1p(x) = log(1 + x)` transformation to normalize variance.
2. `StandardScaler` to ensure all features contribute equally to Euclidean distance metrics.


In [ ]:
X_raw = df[FEATURE_COLS].values
X_log = np.log1p(X_raw)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)
print("Scaled shape:", X_scaled.shape)
print("Means after scaling:", np.round(X_scaled.mean(axis=0), 2))
print("Stds after scaling:", np.round(X_scaled.std(axis=0), 2))


## 4. Modeling & Clustering Evaluation (k = 2 .. 10)
Evaluating K-Means, Agglomerative Clustering (Ward Linkage), and Gaussian Mixture Models across k=2 to 10.


In [ ]:
results = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=25)
    km_labels = km.fit_predict(X_scaled)
    results.append({
        'k': k,
        'Inertia': km.inertia_,
        'Silhouette': silhouette_score(X_scaled, km_labels),
        'Davies_Bouldin': davies_bouldin_score(X_scaled, km_labels),
        'Calinski_Harabasz': calinski_harabasz_score(X_scaled, km_labels)
    })

km_eval_df = pd.DataFrame(results)
print(km_eval_df.to_string(index=False))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(km_eval_df['k'], km_eval_df['Inertia'], marker='o', color='#2b5c8f', linewidth=2)
ax1.axvline(x=3, color='crimson', linestyle='--', label='Selected k=3')
ax1.set_title('Elbow Curve (Inertia vs k)')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.legend()

ax2.plot(km_eval_df['k'], km_eval_df['Silhouette'], marker='s', color='#2ca02c', linewidth=2)
ax2.axvline(x=3, color='crimson', linestyle='--', label='Selected k=3')
ax2.set_title('Silhouette Score vs k')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.legend()
plt.tight_layout()
plt.show()


## 5. Final Model Selection (K-Means, k = 3) & PCA Visualization
k=3 is chosen because it produces balanced, highly actionable segments:
- Cluster 0: Broad-Category High Spenders (Key Accounts)
- Cluster 1: Fresh & Frozen / Horeca Focused (Hospitality)
- Cluster 2: Grocery & Detergents / Retail Specialists (Retail Stores)


In [ ]:
km_final = KMeans(n_clusters=3, random_state=42, n_init=25)
df['Cluster'] = km_final.fit_predict(X_scaled)

CLUSTER_NAMES = {
    0: 'Broad-Category High Spenders',
    1: 'Fresh & Frozen / Horeca Focused',
    2: 'Grocery & Detergents / Retail Specialists'
}
df['Cluster_Name'] = df['Cluster'].map(CLUSTER_NAMES)

# 2D PCA projection
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df['PC1'] = pca_coords[:, 0]
df['PC2'] = pca_coords[:, 1]

var_exp = pca.explained_variance_ratio_ * 100
print(f"PCA Total Explained Variance: {var_exp.sum():.2f}% (PC1: {var_exp[0]:.2f}%, PC2: {var_exp[1]:.2f}%)")

plt.figure(figsize=(9, 6.5))
colors = ['#2b5c8f', '#2ca02c', '#d62728']
for c in range(3):
    sub = df[df['Cluster'] == c]
    plt.scatter(sub['PC1'], sub['PC2'], color=colors[c], label=f'Cluster {c}: {CLUSTER_NAMES[c]} (n={len(sub)})', alpha=0.75, s=60)
plt.title(f'PCA 2D Customer Segments (Explained Variance: {var_exp.sum():.1f}%)')
plt.xlabel(f'PC1 ({var_exp[0]:.1f}%) - Grocery, Milk & Detergents')
plt.ylabel(f'PC2 ({var_exp[1]:.1f}%) - Fresh, Frozen & Delicassen')
plt.legend()
plt.show()


## 6. Cluster Profiling on Original Scale
Calculating mean, median, total spend, and channel/region compositions.


In [ ]:
print("=== Mean Spending per Cluster (m.u.) ===")
print(df.groupby('Cluster_Name')[FEATURE_COLS].mean().round(1))

print("
=== Median Spending per Cluster (m.u.) ===")
print(df.groupby('Cluster_Name')[FEATURE_COLS].median().round(1))

print("
=== Channel Breakdown per Cluster ===")
print(pd.crosstab(df['Cluster_Name'], df['Channel'].map(CHANNEL_MAP), normalize='index').round(3) * 100)


## 7. Strategic Business Insights
1. **Cluster 0 (Broad-Category High Spenders):** Generates >52% of total wholesale revenue. Dedicated account managers, tiered loyalty discounts, and unified fulfillment should be prioritized.
2. **Cluster 1 (Fresh & Frozen / Horeca Focused):** 98.6% Horeca clients. Value fast, reliable cold-chain delivery and seasonal fresh menu bundling.
3. **Cluster 2 (Grocery & Detergents / Retail Specialists):** 60.2% Retail stores. Value volume pallet discounts on non-perishable pantry and cleaning goods.
